In [1]:
import pandas as pd

train_df = pd.read_csv("/kaggle/input/datasets/sathvik2006/annotations/train_annotations.csv")
val_df = pd.read_csv("/kaggle/input/datasets/sathvik2006/annotations/val_annotations.csv")
test_df = pd.read_csv("/kaggle/input/datasets/sathvik2006/annotations/test_annotations.csv")

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

train_df.head()

Train: 39241
Val: 8386
Test: 8453


,image,category,bbox,segmentation
0,161364.jpg,trousers,"[252, 323, 522, 639]","[[432, 324, 339, 353, 250, 393, 290, 532, 346,..."
1,161364.jpg,short sleeve top,"[30, 1, 466, 481]","[[183, 5, 175, 19, 157, 35, 125, 48, 92, 51, 8..."
2,117178.jpg,trousers,"[159, 423, 337, 701]","[[328, 432, 249, 432, 162, 430, 164, 540, 164,..."
3,131919.jpg,trousers,"[158, 381, 366, 650]","[[346, 450, 263, 438, 227, 388, 203, 451, 186,..."
4,131919.jpg,short sleeve top,"[212, 140, 481, 492]","[[429, 214, 407, 230, 378, 229, 354, 210, 353,..."


In [2]:
import pandas as pd



# group labels per image
train_group = train_df.groupby("image")["category"].apply(list).reset_index()
val_group = val_df.groupby("image")["category"].apply(list).reset_index()
test_group = test_df.groupby("image")["category"].apply(list).reset_index()

train_group.head()

,image,category
0,000014.jpg,"[skirt, long sleeve top]"
1,000015.jpg,[long sleeve top]
2,000018.jpg,"[trousers, long sleeve top]"
3,000026.jpg,"[skirt, short sleeve top]"
4,000027.jpg,"[skirt, short sleeve top]"


In [3]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

train_labels = mlb.fit_transform(train_group["category"])
val_labels = mlb.transform(val_group["category"])
test_labels = mlb.transform(test_group["category"])

print(mlb.classes_)

['long sleeve top' 'short sleeve top' 'shorts' 'skirt' 'trousers']


In [4]:
import torch 
labels_matrix = train_labels   # shape [N, C]

pos_counts = labels_matrix.sum(axis=0)
neg_counts = labels_matrix.shape[0] - pos_counts

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pos_weight = torch.tensor(
    neg_counts / (pos_counts + 1e-6),
    dtype=torch.float
).to(device)
print(mlb.classes_)
print("Correct pos_weight:", pos_weight)

['long sleeve top' 'short sleeve top' 'shorts' 'skirt' 'trousers']
Correct pos_weight: tensor([3.0496, 1.0334, 2.9657, 3.6053, 1.6412], device='cuda:0')


In [5]:
from torchvision import transforms

# ImageNet normalization
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Added Gaussian Blur
    transforms.ToTensor(),
    normalize
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    normalize
])

In [6]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import os

class FashionDataset(Dataset):

    def __init__(self, df, labels, img_dir, transform=None):
        self.df = df
        self.labels = labels
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        img_name = self.df.iloc[idx]["image"]
        img_path = os.path.join(self.img_dir, img_name)

        image = Image.open(img_path).convert("RGB")

        label = torch.tensor(self.labels[idx]).float()

        if self.transform:
            image = self.transform(image)

        return image, label

In [7]:
img_dir = "/kaggle/input/datasets/sathvik2006/sample35t/sample_35k/images"

train_dataset = FashionDataset(train_group, train_labels, img_dir, train_tf)
val_dataset = FashionDataset(val_group, val_labels, img_dir, val_tf)
test_dataset = FashionDataset(test_group, test_labels, img_dir, val_tf)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 24500
Val: 5250
Test: 5250


In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32, 5])
